In [ ]:
# This is the code for the LSH project of TDT4305

import configparser  # for reading the parameters file
import sys  # for system errors and printouts
from pathlib import Path  # for paths of files
import os  # for reading the input data
import time  # for timing
import numpy as np  # for creating matrices or arrays
import random  # for randomly generating a and b for hash functions
from itertools import combinations  # for creating candidate pairs in lsh

# Global parameters
parameter_file = "default_parameters.ini"  # the main parameters file
data_main_directory = Path("data")  # the main path were all the data directories are
parameters_dictionary = (
    dict()
)  # dictionary that holds the input parameters, key = parameter name, value = value
document_list = (
    dict()
)  # dictionary of the input documents, key = document id, value = the document


# DO NOT CHANGE THIS METHOD
# Reads the parameters of the project from the parameter file 'file'
# and stores them to the parameter dictionary 'parameters_dictionary'
def read_parameters():
    config = configparser.ConfigParser()
    config.read(parameter_file)
    for section in config.sections():
        for key in config[section]:
            if key == "data":
                parameters_dictionary[key] = config[section][key]
            elif key == "naive":
                parameters_dictionary[key] = bool(config[section][key])
            elif key == "t":
                parameters_dictionary[key] = float(config[section][key])
            else:
                parameters_dictionary[key] = int(config[section][key])


# DO NOT CHANGE THIS METHOD
# Reads all the documents in the 'data_path' and stores them in the dictionary 'document_list'
def read_data(data_path):
    for root, dirs, file in os.walk(data_path):
        for f in file:
            file_path = data_path / f
            doc = open(file_path).read().strip().replace("\n", " ")
            file_id = int(file_path.stem)
            document_list[file_id] = doc


# DO NOT CHANGE THIS METHOD
# Calculates the Jaccard Similarity between two documents represented as sets
def jaccard(doc1, doc2):
    return len(doc1.intersection(doc2)) / float(len(doc1.union(doc2)))


# DO NOT CHANGE THIS METHOD
# Define a function to map a 2D matrix coordinate into a 1D index.
def get_triangle_index(i, j, length):
    if i == j:  # that's an error.
        sys.stderr.write("Can't access triangle matrix with i == j")
        sys.exit(1)
    if j < i:  # just swap the values.
        temp = i
        i = j
        j = temp

    # Calculate the index within the triangular array. Taken from pg. 211 of:
    # http://infolab.stanford.edu/~ullman/mmds/ch6.pdf
    # adapted for a 0-based index.
    k = int(i * (length - (i + 1) / 2.0) + j - i) - 1

    return k


# DO NOT CHANGE THIS METHOD
# Calculates the similarities of all the combinations of documents and returns the similarity triangular matrix
def naive():
    docs_Sets = []  # holds the set of words of each document

    for doc in document_list.values():
        docs_Sets.append(set(doc.split()))

    # Using triangular array to store the similarities, avoiding half size and similarities of i==j
    num_elems = int(len(docs_Sets) * (len(docs_Sets) - 1) / 2)
    similarity_matrix = [0 for x in range(num_elems)]
    for i in range(len(docs_Sets)):
        for j in range(i + 1, len(docs_Sets)):
            similarity_matrix[get_triangle_index(i, j, len(docs_Sets))] = jaccard(
                docs_Sets[i], docs_Sets[j]
            )

    return similarity_matrix


# METHOD FOR TASK 1
# Creates the k-Shingles of each document and returns a list of them


def k_shingles():
    k = 5

    docs_k_shingles = []  # holds the k-shingles of each document

    for doc in document_list.values():
        words = doc.split()

        shingles = [" ".join(words[i : i + k]) for i in range(len(words) - k + 1)]

        docs_k_shingles.append(set(shingles))

    return docs_k_shingles


# METHOD FOR TASK 2
# Creates a signatures set of the documents from the k-shingles list
def signature_set(k_shingles):
    docs_sig_sets = []

    # implement your code here

    # find all the unique shingles
    unique_shingles = list(set.union(*k_shingles))
    print(f"found {len(unique_shingles)} unique shingles")

    """
    initialize the signature matrix with zeros.
    The matrix has the number of rows equal to the number of documents
    and the number of columns equal to the number of documents
    """
    docs_sig_sets = np.zeros((len(k_shingles), len(unique_shingles)))

    # set the signature matrix
    for i in range(len(k_shingles)):
        for j in range(len(unique_shingles)):
            if unique_shingles[j] in k_shingles[i]:
                docs_sig_sets[i, j] = 1
    return docs_sig_sets


# METHOD FOR TASK 3


# A function for generating hash functions
def generate_hash_functions(num_perm, N):
    hash_funcs = []

    p = 2**31 - 1  # A large prime number

    for _ in range(num_perm):
        a = random.randint(1, p - 1)
        b = random.randint(0, p - 1)
        hash_funcs.append(lambda x, a=a, b=b, p=p, N=N: ((a * x + b) % p) % N)

    return hash_funcs


# Creates the minHash signatures after generating hash functions
def minHash(docs_signature_sets, hash_fn):
    min_hash_signatures = []

    # implement your code here

    # initialize the min hash signatures
    min_hash_signatures = np.full((len(hash_fn), len(docs_signature_sets)), np.inf)

    # For each column (feature)
    for col_id in range(len(docs_signature_sets[0])):
        # For each row (document)
        for row_id, row in enumerate(docs_signature_sets):
            # If this feature is present in the document
            if row[col_id] == 1:
                # Update signature with minimum hash value
                for h_id, hash_func in enumerate(hash_fn):
                    hash_value = hash_func(col_id + 1)
                    if hash_value < min_hash_signatures[h_id][row_id]:
                        min_hash_signatures[h_id][row_id] = hash_value

    return min_hash_signatures


# METHOD FOR TASK 4
# Hashes the MinHash Signature Matrix into buckets and find candidate similar documents
def lsh(m_matrix):
    candidates = []  # list of candidate sets of documents for checking similarity

    # implement your code here
    number_of_rows = m_matrix.shape[1]
    bands = np.array_split(m_matrix, parameters_dictionary["b"], axis=0)
    
    # Assign each document to buckets in different bands
    for band in bands:
        buckets = {}
        for document_id in range(number_of_rows):
            signature = tuple(band[:, document_id].astype(int))
            if signature in buckets:
                buckets[signature].append(document_id)
            else:
                buckets[signature] = [document_id]

        #Candidate pairs
        for documents in buckets.values():
            if len(documents) > 1:
                for pair in combinations(documents, 2):
                    candidates.append(pair)
                    
    return candidates

# METHOD FOR TASK 5
# Calculates the similarities of the candidate documents
def candidates_similarities(candidate_docs, min_hash_matrix):
    similarity_dict = []

    # implement your code here

    number_of_hash_fns = min_hash_matrix.shape[0]
    
    #for each 
    for document1_id, document2_id in candidate_docs:
        
        doc1 = min_hash_matrix[document1_id]
        doc2 = min_hash_matrix[document2_id]
        
        
        similarity = np.sum(doc1 == doc2) / number_of_hash_fns
        similarity_dict.append((document1_id, document2_id), similarity))

    return similarity_dict


In [2]:
# DO NOT CHANGE THIS METHOD
# The main method where all code starts
# Reading the parameters
read_parameters()

# Reading the data
print("Data reading...")
data_folder = data_main_directory / parameters_dictionary["data"]
t0 = time.time()
read_data(data_folder)
document_list = {k: document_list[k] for k in sorted(document_list)}
t1 = time.time()
print(len(document_list), "documents were read in", t1 - t0, "sec\n")

# Naive
naive_similarity_matrix = []
if parameters_dictionary["naive"]:
    print("Starting to calculate the similarities of documents...")
    t2 = time.time()
    naive_similarity_matrix = naive()
    t3 = time.time()
    print(
        "Calculating the similarities of",
        len(naive_similarity_matrix),
        "combinations of documents took",
        t3 - t2,
        "sec\n",
    )

# k-Shingles
print("Starting to create all k-shingles of the documents...")
t4 = time.time()
all_docs_k_shingles = k_shingles()
t5 = time.time()
print("Representing documents with k-shingles took", t5 - t4, "sec\n")

# signatures sets
print("Starting to create the signatures of the documents...")
t6 = time.time()
signature_sets = signature_set(all_docs_k_shingles)
t7 = time.time()
print("Signatures representation took", t7 - t6, "sec\n")

# Permutations
print("Starting to simulate the MinHash Signature Matrix...")
t8 = time.time()
hash_fn = generate_hash_functions(
    parameters_dictionary["permutations"], len(signature_sets)
)
min_hash_signatures = minHash(signature_sets, hash_fn)
t9 = time.time()
print("Simulation of MinHash Signature Matrix took", t9 - t8, "sec\n")



Data reading...
2225 documents were read in 0.22594690322875977 sec

Starting to calculate the similarities of documents...
Calculating the similarities of 2474200 combinations of documents took 35.38093113899231 sec

Starting to create all k-shingles of the documents...
Representing documents with k-shingles took 0.2673821449279785 sec

Starting to create the signatures of the documents...
found 759983 unique shingles
Signatures representation took 77.51585102081299 sec

Starting to simulate the MinHash Signature Matrix...
Simulation of MinHash Signature Matrix took 247.42833304405212 sec



In [7]:
# LSH
print("Starting the Locality-Sensitive Hashing...")
t10 = time.time()
candidate_docs = lsh(min_hash_signatures)
t11 = time.time()
print("LSH took", t11 - t10, "sec\n")

# Return the over t similar pairs
print(
    "Starting to get the pairs of documents with over ",
    parameters_dictionary["t"],
    "% similarity...",
)
t14 = time.time()
true_pairs = candidates_similarities(candidate_docs, min_hash_signatures)
t15 = time.time()
print(f"The total number of candidate pairs from LSH: {len(candidate_docs)}")
print(f"The total number of true pairs from LSH: {len(true_pairs)}")
print(
    f"The total number of false positives from LSH: {len(candidate_docs) - len(true_pairs)}"
)

if parameters_dictionary["naive"]:
    print("Naive similarity calculation took", t3 - t2, "sec")

print("LSH process took in total", t14 - t15, "sec")

print("The pairs of documents are:\n")
for p in true_pairs:
    print(
        f"LSH algorith reveals that the BBC article {list(p.keys())[0][0] + 1}.txt and {list(p.keys())[0][1] + 1}.txt \
            are {round(list(p.values())[0], 2) * 100}% similar"
    )

    print("\n")

Starting the Locality-Sensitive Hashing...
LSH took 0.07818770408630371 sec

Starting to get the pairs of documents with over  0.6 % similarity...
The total number of candidate pairs from LSH: 604
The total number of true pairs from LSH: 0
The total number of false positives from LSH: 604
Naive similarity calculation took 35.38093113899231 sec
LSH process took in total -2.574920654296875e-05 sec
The pairs of documents are:

